# 

# Alternative span architecture

In [98]:
%pip install -q evaluate

Note: you may need to restart the kernel to use updated packages.


In [99]:
import pandas as pd
import torch
from pathlib import Path
import json
import time


print("GPU available:", torch.cuda.is_available() or torch.backends.mps.is_available())
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset
from datasets import Dataset

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering, Trainer, TrainingArguments
from transformers import AutoModelForTokenClassification


from evaluate import load as load_metric

import numpy as np
from scipy.special import log_softmax

import re
import string

from transformers import set_seed
set_seed(42)
SEED=42

GPU available: True


In [100]:
# Paths of models 
SMOKE_TEST = False
BIO_MODEL_DIR = f"multilingual-qa-bio-seed{SEED}" + ("-smoke" if SMOKE_TEST else "")
RESULTS_DIR = Path(BIO_MODEL_DIR, "results")
POINTER_MODEL_DIR = "multilingual-qa"  # Week 40-41 model, seed 42 QA model 

In [101]:
import sys
import transformers

print("Python:", sys.version.split()[0])
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "MPS" if torch.backends.mps.is_available() else "None")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.version.cuda)

Python: 3.13.15
GPU: MPS
PyTorch: 2.14.1
Transformers: 5.17.0
CUDA: None


# Load dataset

Loading dataset, and setup a tokenizer.

In [102]:
dataset = load_dataset("coastalcph/tydi_xor_rc")


df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

# Selected languages
langlst = ['ko','ar','te']

# Training and Validation data for selected languages
df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]

#Training and Validation data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]


saved_model_exists = all(
    Path(BIO_MODEL_DIR, filename).is_file()
    for filename in ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json"]
)
model_source = BIO_MODEL_DIR if saved_model_exists else "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(
    model_source, use_fast=True, local_files_only=saved_model_exists
)

# BIO labels

In [103]:
# Windows likes the orinal model. 
# Difference is every context token gets a label, 
# specials，padding and quetsion tokens gets -100(ignored by the loss). 
# No answer in the window: all O
LABELS = ["O", "B-ANS", "I-ANS"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for i, label in enumerate(LABELS)}
IGNORE_INDEX = -100

In [104]:
# Same as QA model 
def tokenize_windows(questions, contexts):
    return tokenizer(
        questions, contexts,
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

def context_offsets(tokens, window):
    sequence_ids = tokens.sequence_ids(window)
    return [
        offset if sequence_ids[j] == 1 else None
        for j, offset in enumerate(tokens["offset_mapping"][window])
    ]

In [105]:
def character_span_to_bio(context, offsets, answer_start=None, answer_text=""):
    # Default
    labels = ["O" if offset is not None else None for offset in offsets]
 
  
    if answer_start is None or answer_text == "":
        return labels
 
    answer_end = answer_start + len(answer_text)
    if context[answer_start:answer_end] != answer_text:
        raise ValueError("The supplied answer text does not match the character span")
 
    #Select only context tokens 
    window_offsets = []
    for offset in offsets:
        if offset is not None:
            window_offsets.append(offset)
 
    if (not window_offsets
            or window_offsets[0][0] > answer_start
            or window_offsets[-1][1] < answer_end):
        return labels
 

    covered = [
        index
        for index, offset in enumerate(offsets)
        if offset is not None and offset[0] < answer_end and offset[1] > answer_start
    ]
    if not covered:
        raise ValueError("The answer does not overlap any token")
 
    labels[covered[0]] = "B-ANS"
    for index in covered[1:]:
        labels[index] = "I-ANS"
    return labels
 
 
def bio_to_character_span(context, offsets, labels):
    if len(offsets) != len(labels):
        raise ValueError("Offsets and labels must have the same length")
    if not set(labels) <= {"O", "B-ANS", "I-ANS", None}:
        raise ValueError("Only O, B-ANS and I-ANS labels are supported")
 
    answer_indices = [
        index for index, label in enumerate(labels) if label in ("B-ANS", "I-ANS")
    ]
    if not answer_indices:
        return None, ""
 
    start_index = answer_indices[0]
    expected_indices = list(range(start_index, start_index + len(answer_indices)))
    expected_labels = ["B-ANS"] + ["I-ANS"] * (len(answer_indices) - 1)
    if answer_indices != expected_indices:
        raise ValueError("The labels contain multiple or non-contiguous answer spans")
    if [labels[index] for index in answer_indices] != expected_labels:
        raise ValueError("The answer span must begin with B-ANS and continue with I-ANS")
 
    start = offsets[answer_indices[0]][0]
    end = offsets[answer_indices[-1]][1]
    return start, context[start:end]


In [106]:
def prepare_examples(examples):
    tokens = tokenize_windows(examples["question"], examples["context"])
    example_numbers = tokens.pop("overflow_to_sample_mapping")
    all_labels = []
 
    for i, example_number in enumerate(example_numbers):
        answerable = examples["answerable"][example_number]
        answer_start = int(examples["answer_start"][example_number]) if answerable else None
        answer = examples["answer"][example_number] if answerable else ""
 
        labels = character_span_to_bio(
            examples["context"][example_number],
            context_offsets(tokens, i),
            answer_start,
            answer,
        )
        all_labels.append([
            label2id[label] if label is not None else IGNORE_INDEX
            for label in labels
        ])
 
    tokens.pop("offset_mapping")
    tokens["labels"] = all_labels
    return tokens

# Training BIO Model

In [107]:
if saved_model_exists:
    model = AutoModelForTokenClassification.from_pretrained(
        BIO_MODEL_DIR, local_files_only=True
    )
    training_args = TrainingArguments(
        output_dir=BIO_MODEL_DIR,
        fp16=torch.cuda.is_available(),
        per_device_eval_batch_size=8,
        seed=SEED,
        full_determinism=True,
        report_to="none",
    )
    bio_trainer = Trainer(model=model, args=training_args)
    print("Loaded saved BIO model. Skipping training.")
 
else:
    model = AutoModelForTokenClassification.from_pretrained(
        "bert-base-multilingual-cased",
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id,
    )
    train_examples = Dataset.from_pandas(df_train_filtered, preserve_index=False)
    train_features = train_examples.map(
        prepare_examples,
        batched=True,
        remove_columns=train_examples.column_names,
    )
 
    training_args = TrainingArguments(
        output_dir=BIO_MODEL_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=8,
        fp16=torch.cuda.is_available(),
        learning_rate=3e-5,
        seed=SEED,
        full_determinism=True,
        save_strategy="no",
        report_to="none",
    )
    bio_trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_features,
    )
    training_start = time.time()
    bio_trainer.train()
    training_minutes = (time.time() - training_start) / 60
 
    bio_trainer.save_model(BIO_MODEL_DIR)
    tokenizer.save_pretrained(BIO_MODEL_DIR)
    print("Saved final model and tokenizer.")
 
    # Resource record for the reproducibility appendix (printed only: copy it after training).
    training_info = {
        "seed": SEED,
        "training_examples": len(train_examples),
        "training_windows": len(train_features),
        "epochs": training_args.num_train_epochs,
        "training_minutes": round(training_minutes, 1),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "MPS" if torch.backends.mps.is_available() else "None",
    }
    print(training_info)
 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded saved BIO model. Skipping training.


# Validadtion & Calculating spans scores

In [108]:
def prepare_validation(examples):
    tokens = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
 
    example_numbers = tokens.pop("overflow_to_sample_mapping")
    tokens["example_index"] = example_numbers
 
    for i in range(len(tokens["input_ids"])):
        sequence_ids = tokens.sequence_ids(i)
        tokens["offset_mapping"][i] = [
            offset if sequence_ids[j] == 1 else None
            for j, offset in enumerate(tokens["offset_mapping"][i])
        ]
 
    return tokens
 
MAX_ANSWER_LENGTH = 30

#Function that calculates the span score by pairs.
def score_spans(features, start_logits, end_logits, bio_logits, contexts):
    n = len(contexts)
    pointer_null = np.full(n, np.inf)
    pointer_span = np.full(n, -np.inf)
    pointer_best = [""] * n
    bio_span = np.full(n, -np.inf)
    bio_best = [""] * n
    log_probs = log_softmax(np.asarray(bio_logits, dtype=np.float32), axis=-1)
 
    for feature_index, feature in enumerate(features):
        example_index = feature["example_index"]
        offsets = feature["offset_mapping"]
        starts = start_logits[feature_index]
        ends = end_logits[feature_index]
        # Gain of labelling a token B (or I) instead of O.
        begin_gain = log_probs[feature_index][:, label2id["B-ANS"]] - log_probs[feature_index][:, label2id["O"]]
        inside_gain = log_probs[feature_index][:, label2id["I-ANS"]] - log_probs[feature_index][:, label2id["O"]]
 
        cls_index = feature["input_ids"].index(tokenizer.cls_token_id)
        pointer_null[example_index] = min(
            pointer_null[example_index],
            float(starts[cls_index] + ends[cls_index]),
        )
 
        for start, start_offset in enumerate(offsets):
            if start_offset is None or start_offset[1] <= start_offset[0]:
                continue
 
            bio_score = float(begin_gain[start])
            for end in range(start, min(start + MAX_ANSWER_LENGTH, len(offsets))):
                end_offset = offsets[end]
                if end_offset is None:
                    break
                if end > start:
                    bio_score += float(inside_gain[end])
                if end_offset[1] <= end_offset[0]:
                    continue
 
                answer = contexts[example_index][start_offset[0]:end_offset[1]]
                pointer_score = float(starts[start] + ends[end])
                if pointer_score > pointer_span[example_index]:
                    pointer_span[example_index] = pointer_score
                    pointer_best[example_index] = answer
                if bio_score > bio_span[example_index]:
                    bio_span[example_index] = bio_score
                    bio_best[example_index] = answer
 
    pointer_scores = pointer_null - pointer_span
    bio_scores = 0.0 - bio_span
    return pointer_scores, pointer_best, bio_scores, bio_best
 

In [109]:
pointer_model = AutoModelForQuestionAnswering.from_pretrained(
    POINTER_MODEL_DIR, local_files_only=True
)
pointer_args = TrainingArguments(
    output_dir=POINTER_MODEL_DIR,
    fp16=torch.cuda.is_available(),
    per_device_eval_batch_size=8,
    seed=42,
    full_determinism=True,
    report_to="none",
)
pointer_trainer = Trainer(model=pointer_model, args=pointer_args)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [110]:
validation_examples = Dataset.from_pandas(
    df_validation_filtered, preserve_index=False
)
# One batch keeps example_index aligned with the full validation dataframe.
validation_features = validation_examples.map(
    prepare_validation,
    batched=True,
    batch_size=len(validation_examples),
    remove_columns=validation_examples.column_names,
)
model_inputs = validation_features.remove_columns(
    ["offset_mapping", "example_index"]
)
start_logits, end_logits = pointer_trainer.predict(model_inputs).predictions
bio_logits = bio_trainer.predict(model_inputs).predictions
 
pointer_scores, pointer_best_answers, bio_scores, bio_best_answers = score_spans(
    validation_features, start_logits, end_logits, bio_logits,
    list(validation_examples["context"]),
)

Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

/opt/anaconda3/envs/nlpproj/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


# Thredshold selection


In [111]:
def select_threshold(scores, gold_answerable):
    thresholds = [scores.min() - 1] + list(np.unique(scores))
    best_threshold = thresholds[0]
    best_macro_f1 = -1
 
    for threshold in thresholds:
        predicted_answerable = scores <= threshold
        macro_f1 = f1_score(
            gold_answerable,
            predicted_answerable,
            labels=[False, True],
            average="macro",
            zero_division=0,
        )
 
        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_threshold = threshold
 
    return float(best_threshold)
 
gold_answerable = df_validation_filtered["answerable"].to_numpy(dtype=bool)
pointer_threshold = select_threshold(pointer_scores, gold_answerable)
bio_threshold = select_threshold(bio_scores, gold_answerable)
 
print("Pointer threshold:", pointer_threshold)
print("BIO threshold:", bio_threshold)
 
pointer_predicted_answers = [
    answer if score <= pointer_threshold else ""
    for answer, score in zip(pointer_best_answers, pointer_scores)
]
bio_predicted_answers = [
    answer if score <= bio_threshold else ""
    for answer, score in zip(bio_best_answers, bio_scores)
]

Pointer threshold: -0.3616856336593628
BIO threshold: 0.6258541345596313


In [112]:
def normalize_answer(text):
    text = text.lower()
    text = "".join(c for c in text if c not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())
 
squad_metric = load_metric("squad_v2")
gold_text = df_validation_filtered["answer"].where(df_validation_filtered["answerable"], "")
pointer_correct = (
    pd.Series(pointer_predicted_answers, index=df_validation_filtered.index).map(normalize_answer)
    == gold_text.map(normalize_answer)
)
results = []
 
for model_name, predicted_answers in [
    ("Pointer", pointer_predicted_answers),
    ("BIO", bio_predicted_answers),
]:
    evaluation = df_validation_filtered.copy()
    evaluation["prediction"] = predicted_answers
    evaluation["pred_answerable"] = evaluation["prediction"] != ""
    evaluation["correct"] = evaluation["prediction"].map(normalize_answer) == gold_text.map(normalize_answer)
    evaluation["pointer_correct"] = pointer_correct
 
    for language in ["overall", "ar", "ko", "te"]:
        group = evaluation if language == "overall" else evaluation[evaluation["lang"] == language]
        formatted_predictions = []
        gold = []
 
        for example_id, example in group.iterrows():
            formatted_predictions.append({
                "id": str(example_id),
                "prediction_text": example["prediction"],
                # Encode the decision already made, not a calibrated probability.
                "no_answer_probability": float(not example["pred_answerable"]),
            })
            gold.append({
                "id": str(example_id),
                "answers": {
                    "text": [example["answer"]] if example["answerable"] else [],
                    "answer_start": [int(example["answer_start"])] if example["answerable"] else [],
                },
            })
 
        squad_results = squad_metric.compute(
            references=gold, predictions=formatted_predictions
        )
        report = classification_report(
            group["answerable"], group["pred_answerable"],
            labels=[False, True], target_names=["unanswerable", "answerable"],
            output_dict=True, zero_division=0,
        )
 
        row = {
            "model": model_name,
            "language": language,
            "N": len(group),
            "SQuAD_v2_EM": squad_results["exact"],
            "SQuAD_v2_token_F1": squad_results["f1"],
            "HasAns_EM": squad_results["HasAns_exact"],
            "answerability_macro_F1": 100 * report["macro avg"]["f1-score"],
            "wrong_to_correct": (~group["pointer_correct"] & group["correct"]).sum(),
            "correct_to_wrong": (group["pointer_correct"] & ~group["correct"]).sum(),
        }
        for label in ["unanswerable", "answerable"]:
            row[f"{label}_P"] = 100 * report[label]["precision"]
            row[f"{label}_R"] = 100 * report[label]["recall"]
            row[f"{label}_F1"] = 100 * report[label]["f1-score"]
        results.append(row)
 
# Subtract the pointer score for the same language.
pointer_rows = {row["language"]: row for row in results if row["model"] == "Pointer"}
for row in results:
    baseline = pointer_rows[row["language"]]
    row["EM_change"] = row["HasAns_EM"] - baseline["HasAns_EM"]
    row["macro_F1_change"] = row["answerability_macro_F1"] - baseline["answerability_macro_F1"]
 
results_table = pd.DataFrame(results)
display(results_table.round(2))

,model,language,N,SQuAD_v2_EM,SQuAD_v2_token_F1,HasAns_EM,answerability_macro_F1,wrong_to_correct,correct_to_wrong,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1,EM_change,macro_F1_change
0,Pointer,overall,1155,57.49,66.41,55.30,84.32,0,0,75.32,70.73,72.96,95.20,96.17,95.68,0.00,0.00
1,Pointer,ar,415,62.41,71.11,57.58,95.25,0,0,87.72,96.15,91.74,99.44,98.07,98.75,0.00,0.00
2,Pointer,ko,356,57.58,67.80,56.97,79.76,0,0,56.52,68.42,61.90,98.20,97.03,97.61,0.00,0.00
3,Pointer,te,384,52.08,60.02,50.52,76.66,0,0,71.62,56.99,63.47,87.10,92.78,89.85,0.00,0.00
4,BIO,overall,1155,51.95,61.30,51.77,70.97,61,125,48.33,53.05,50.58,92.10,90.62,91.35,-3.53,-13.35
5,BIO,ar,415,57.83,67.66,52.89,84.14,18,37,60.76,92.31,73.28,98.81,91.46,94.99,-4.68,-11.11
6,BIO,ko,356,53.65,63.80,52.52,69.53,17,31,31.82,73.68,44.44,98.40,91.10,94.61,-4.45,-10.23
7,BIO,te,384,44.01,52.11,49.48,58.58,26,57,43.86,26.88,33.33,79.20,89.00,83.82,-1.03,-18.09


# Comparison 

In [113]:
comparison = df_validation_filtered.copy()
comparison["pointer"] = pointer_predicted_answers
comparison["BIO"] = bio_predicted_answers
both_answered = comparison[
    comparison["answerable"] & (comparison["pointer"] != "") & (comparison["BIO"] != "")
]
 
span_rows = []
for model_name, column in [("Pointer", "pointer"), ("BIO", "BIO")]:
    counts = {"exact": 0, "contains gold + extra words": 0, "inside gold": 0, "other": 0}
    for prediction, answer in zip(both_answered[column], both_answered["answer"]):
        predicted, expected = normalize_answer(prediction), normalize_answer(answer)
        if predicted == expected:
            counts["exact"] += 1
        elif f" {expected} " in f" {predicted} ":
            counts["contains gold + extra words"] += 1
        elif f" {predicted} " in f" {expected} ":
            counts["inside gold"] += 1
        else:
            counts["other"] += 1
    span_rows.append({
        "model": model_name,
        "N": len(both_answered),
        **counts,
        "mean predicted words": np.mean([len(p.split()) for p in both_answered[column]]),
        "mean gold words": np.mean([len(a.split()) for a in both_answered["answer"]]),
    })
 
display(pd.DataFrame(span_rows).round(2))

,model,N,exact,contains gold + extra words,inside gold,other,mean predicted words,mean gold words
0,Pointer,882,536,96,64,186,2.96,3.04
1,BIO,882,505,74,87,216,2.54,3.04
